In [7]:
import arcade
import numpy as np
import import_ipynb
from notebook import smart_move, won_game, get_winning_cells, player_turn

In [8]:
SCREEN_WIDTH = 600
SCREEN_HEIGHT = 600
SCREEN_TITLE = "Tic Tac Toe"
LINES = ((200,0),(200,600),(400,0),(400,600),(0,200),(600,200),(0,400),(600,400))

In [9]:
class TicTacToeGame(arcade.Window):
    def __init__(self, width, height, title):
        super().__init__(width, height, title)
        arcade.set_background_color(arcade.color.LIGHT_BLUE)

        self.state="menu"
        self.player_symbol = None
        self.text_objects: list[list[arcade.Text | None]] = [[None]*3 for _ in range(3)]

        self.board = np.zeros((3,3), dtype=int)
        self.game_over = False
        self.winning_cells = None

        # Menu Text
        self.title_text = arcade.Text("Tic Tac Toe", 300, 500, arcade.color.BLACK, 40, anchor_x="center", anchor_y="center")

        self.choose_text = arcade.Text("Click Your Letter:", 300, 400,arcade.color.BLACK, 20, anchor_x="center")

        self.x_text = arcade.Text("X", 200, 300,arcade.color.BLACK, 30,anchor_x="center", anchor_y="center")

        self.o_text = arcade.Text("O", 400, 300,arcade.color.BLACK, 30,anchor_x="center", anchor_y="center")

        self.start_text = arcade.Text("Click To Start", 300, 200,arcade.color.BLACK, 25,anchor_x="center", anchor_y="center")

        self.result_text = arcade.Text("", 300, 550,arcade.color.BLACK, 30,anchor_x="center")

        self.restart = arcade.Text("Restart Game",300, 350, arcade.color.BLACK, 30,anchor_x="center")

    def make_move(self, row, col, symbol):
        text = "X" if symbol == 1 else "O"
        color = arcade.color.RED if symbol == 1 else arcade.color.BLUE

        self.text_objects[row][col] = arcade.Text(
            text,
            col * 200 + 100,
            row * 200 + 100,
            color,
            100,
            anchor_x="center",
            anchor_y="center"
        )

    def start_game(self):
        self.board = np.zeros((3,3), dtype=int)
        self.text_objects = [[None]*3 for _ in range(3)]
        self.game_over = False
        self.winning_cells = None

        # If player chose O, computer starts
        if self.player_symbol == -1:
            move = smart_move(self.board, 1, -1)
            if move:
                r, c = move
                self.make_move(r, c, 1)
                self.board[r][c] = 1

    def end_game(self, message):
        self.game_over = True
        self.result_text.text = message

    def on_draw(self):
        self.clear()

        if self.state == "menu":
            self.title_text.draw()
            self.choose_text.draw()

            arcade.draw_lbwh_rectangle_filled(150,270,100,50,arcade.color.RED if self.player_symbol == 1 else arcade.color.GRAY)

            self.x_text.draw()

            arcade.draw_lbwh_rectangle_filled(350,270,100,50,arcade.color.BLUE if self.player_symbol == -1 else arcade.color.GRAY)

            self.o_text.draw()

            if self.player_symbol is not None:
                arcade.draw_lbwh_rectangle_filled(200,170,200,60,arcade.color.GREEN)
                self.start_text.draw()

            return


        arcade.draw_lines(LINES,color=arcade.color.WHITE,line_width=10)

        for row in range(3):
            for col in range(3):
                if self.text_objects[row][col]:
                    self.text_objects[row][col].draw()

        if self.winning_cells:
            (r1,c1),_,(r3,c3) = self.winning_cells

            a1 = c1 * 200 + 100
            b1 = r1 * 200 + 100

            a3 = c3 * 200 + 100
            b3 = r3 * 200 + 100

            arcade.draw_line(a1,b1,a3,b3,arcade.color.BLACK,10)

        if self.game_over:
            arcade.draw_lbwh_rectangle_filled(0,0,600,600,(255,255,255,100))
            arcade.draw_lbwh_rectangle_filled(185,325,230,75,(40,255,0,225))
            self.result_text.draw()
            self.restart.draw()
            return

    def on_mouse_press(self, x, y, button, modifiers):
        if self.state =="menu":
            if 150 < x < 250 and 275 < y < 325:
                self.player_symbol = 1

            if 350 < x < 450 and 275 < y < 325:
                self.player_symbol = -1

            if self.player_symbol is not None:
                if 225< x < 375 and 170 < y < 230:
                    self.state = "game"
                    self.start_game()

            return

        if self.game_over:
            if 180 < x < 420 and 325 < y < 400:
                self.state = "menu"
                self.player_symbol = None
                self.start_game()
            return

        if self.state != "game":
            return

        # Prevent player from going first if they chose O
        if self.player_symbol == -1 and np.all(self.board == 0):
            return

        col = int(x // 200)
        row = int(y // 200)

        if 0 <= row < 3 and 0 <= col < 3:
            if self.board[row][col] == 0:

                comp_move, result = player_turn(self.board, row, col, self.player_symbol)

                # Update player text
                self.make_move(row, col, self.player_symbol)

                # Update computer move (if exists)
                if comp_move:
                    r, c = comp_move
                    self.make_move(r, c, -self.player_symbol)

                # Handle results
                if result == "player":
                    self.winning_cells = get_winning_cells(self.board)
                    self.end_game("YOU WIN!")
                    return

                elif result == "computer":
                    self.winning_cells = get_winning_cells(self.board)
                    self.end_game("YOU LOSE!")
                    return

                elif result == "draw":
                    self.end_game("ITS A DRAW!")
                    return

def main():
    TicTacToeGame(SCREEN_WIDTH, SCREEN_HEIGHT, SCREEN_TITLE)

    arcade.run()

if __name__ == "__main__":
    main()